# BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

In [1]:
# Importing Necessary Libraries:

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scrapper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [2]:
# Initializing Constants:

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print('API key found and Looks Good!!')

else:
    print('There might be a Problem with API Key!')

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key found and Looks Good!!


## First step: Have GPT-5-nano figure out which links are relevant:

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

In [4]:
# Defining System Prompt:

link_system_prompt = '''
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
'''

In [5]:
# Defining A Function for Creating User Prompt:

def get_links_user_prompt(url):

    user_prompt = f'''
    Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

    '''

    links = fetch_website_links(url)
    user_prompt += '\n'.join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt('https://edwarddonner.com'))


    Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

    https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwar

In [9]:
# Defining a Function to Get Relevant Links from a WebPage using LLM API Call:

def select_relevant_links(url):

    print(f'Selecting relevant links for {url} by calling {MODEL}')
    response = openai.chat.completions.create(
        model= MODEL,
        messages= [
            {'role': 'system', 'content': link_system_prompt},
            {'role': 'user', 'content': get_links_user_prompt(url)}
        ],
        response_format= {'type': 'json_object'}
    )

    result = response.choices[0].message.content

    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")

    return links

In [10]:
select_relevant_links('https://huggingface.co')

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'product endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Chinese community',
   'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: Making the brochure!

Assemble all the details into another prompt to GPT-5-nano.

In [12]:
# Defining a Function to Fetch Webpage Details and All Relevant Links from it:

def fetch_page_and_all_relevant_links(url):

    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page: \n\n{contents}\n\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])

    return result

In [13]:
print(fetch_page_and_all_relevant_links('https://huggingface.co'))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links
## Landing Page: 

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
fishaudio/s2-pro
Updated
5 days ago
•
5.41k
•
506
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
9 days ago
•
67.6k
•
737
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
6 days ago
•
86.7k
•
298
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
13 days ago
•
238k
•
479
Lightricks/LTX-2.3
Updated
about 21 hours ago
•
597k
•
640
Browse 2M+ models
Spaces
Running
on
Zero


In [14]:
# Defining Final Brochure System Prompt:

brochure_system_prompt = '''
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
'''

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = '''
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# '''

In [15]:
# Defining Brochure User Prompt:

def get_brochure_user_prompt(company_name, url):

    user_prompt = f'''
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
'''

    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5000] # Truncate if more than 5,000 characters
    return user_prompt